# Production Patterns for Tool Use

A notebook agent loop is not production. Real deployments need **factory patterns** (compile once at startup), a **thin service layer**, **observability hooks**, **cost guards**, and a clear path from HTTP request to tool execution.

This capstone assembles those patterns for **ShopCo Order Support** — the same scenario used across tool-calling, security, and MAS vocabulary demos. Everything runs in plain Python with an offline model stand-in; flip `USE_LIVE_LLM = True` when you have an OpenAI key.

**What you will build:**

- `build_tool_registry()` — single startup registry with schemas and risk metadata
- `build_support_agent()` — reusable agent loop with hooks and guards
- `ShopCoAgentService` — stable `run_support_agent()` contract for HTTP and workers
- `SupportChatAPI` — simulated HTTP layer (`/health`, `/support/chat`)
- Tracing, cost circuit breakers, rate limits, and a deployment checklist

In [ ]:
"""
ShopCo Order Support — production patterns capstone.
"""

import json
import os
import re
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

USE_LIVE_LLM = False
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)

---

## 1. Production Architecture

```
┌─────────────────────────────────────────────────────────────┐
│  L7  HTTP API          GET /health  POST /support/chat      │
├─────────────────────────────────────────────────────────────┤
│  L6  Service layer     ShopCoAgentService.run_support_agent │
├─────────────────────────────────────────────────────────────┤
│  L5  Agent factory     build_support_agent() — compile once │
├─────────────────────────────────────────────────────────────┤
│  L4  Agent loop        model ↔ tools (ReAct)                │
├─────────────────────────────────────────────────────────────┤
│  L3  Tool registry     build_tool_registry() + risk tiers   │
├─────────────────────────────────────────────────────────────┤
│  L2  Integrations      ORDERS, POLICIES, ticket store       │
├─────────────────────────────────────────────────────────────┤
│  L1  Observability     TRACE_STORE, hooks, structured logs  │
└─────────────────────────────────────────────────────────────┘
```

**Key rule:** compile the registry and agent **once at startup**, not per request. HTTP handlers stay thin — they call the service layer and return JSON.

---

## 2. Configuration — Settings from Environment

Load secrets and limits from environment variables. Never hardcode API keys in graph modules or tool definitions.

In [ ]:
@dataclass
class AppSettings:
    openai_api_key: str = field(default_factory=lambda: OPENAI_API_KEY)
    default_model: str = "gpt-4o-mini"
    max_iterations: int = 6
    max_tool_calls_per_run: int = 20
    max_cost_usd: float = 0.50
    cost_per_iteration_usd: float = 0.002
    rate_limit_per_minute: int = 30
    tracing_enabled: bool = True
    tracing_project: str = "shopco-support-prod"

    @classmethod
    def from_env(cls) -> "AppSettings":
        return cls(
            openai_api_key=os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY),
            default_model=os.environ.get("APP_MODEL", "gpt-4o-mini"),
            max_iterations=int(os.environ.get("APP_MAX_ITERATIONS", "6")),
            max_tool_calls_per_run=int(os.environ.get("APP_MAX_TOOL_CALLS", "20")),
            max_cost_usd=float(os.environ.get("APP_MAX_COST_USD", "0.50")),
            tracing_project=os.environ.get("APP_TRACING_PROJECT", "shopco-support-prod"),
        )


settings = AppSettings.from_env()
print(f"model={settings.default_model} | max_iter={settings.max_iterations} | cost_cap=${settings.max_cost_usd}")

---

## 3. L2 Integrations — ShopCo Backend

Tools wrap real integrations. Here we use in-memory stores; in production these call your order API, policy CMS, and ticketing system.

In [ ]:
ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICY_DOCS = [
    {"id": "ship-01", "text": "Standard shipping 3-5 business days."},
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
]

TICKETS: list[dict[str, Any]] = []


def _status_ok(payload: Any) -> str:
    return f"[STATUS: ok] {json.dumps(payload, default=str)}"


def _status_error(msg: str) -> str:
    return f"[STATUS: error] {msg}"


def lookup_order(order_id: str) -> str:
    oid = order_id.upper().strip()
    if not re.match(r"^ORD-\d{4}$", oid):
        return _status_error(f"Invalid order_id '{order_id}'")
    order = ORDERS.get(oid)
    if not order:
        return _status_error(f"Order {oid} not found")
    return _status_ok({"order_id": oid, **order})


def search_policy(query: str) -> str:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    hits = [d for d in POLICY_DOCS if any(t in d["text"].lower() for t in terms)]
    return _status_ok(hits or POLICY_DOCS[:1])


def create_support_ticket(customer_email: str, subject: str, body: str) -> str:
    if "@" not in customer_email:
        return _status_error("Invalid email address")
    ticket = {
        "ticket_id": f"TKT-{uuid.uuid4().hex[:6].upper()}",
        "customer": customer_email,
        "subject": subject[:120],
        "body": body[:500],
        "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    TICKETS.append(ticket)
    return _status_ok(ticket)


print(lookup_order("ORD-1001"))
print(search_policy("returns")[:80], "...")

---

## 4. Tool Registry Factory

`build_tool_registry()` is the single place to register tools, JSON schemas, and risk metadata. Call it **once at application startup**.

In [ ]:
class RiskTier(str, Enum):
    READ = "read"
    WRITE = "write"


@dataclass
class ToolMeta:
    risk: RiskTier
    description: str
    requires_approval: bool = False


@dataclass
class ToolRegistry:
    implementations: dict[str, Callable[..., str]]
    schemas: list[dict[str, Any]]
    metadata: dict[str, ToolMeta]

    def get(self, name: str) -> Callable[..., str] | None:
        return self.implementations.get(name)

    def names(self) -> list[str]:
        return list(self.implementations.keys())


def build_tool_registry() -> ToolRegistry:
    """Factory — call once at startup."""
    implementations = {
        "lookup_order": lookup_order,
        "search_policy": search_policy,
        "create_support_ticket": create_support_ticket,
    }
    metadata = {
        "lookup_order": ToolMeta(RiskTier.READ, "Look up order by ORD-#### id"),
        "search_policy": ToolMeta(RiskTier.READ, "Search shipping/returns policies"),
        "create_support_ticket": ToolMeta(
            RiskTier.WRITE, "Create a support ticket", requires_approval=False
        ),
    }
    schemas = [
        {
            "type": "function",
            "function": {
                "name": "lookup_order",
                "description": metadata["lookup_order"].description,
                "parameters": {
                    "type": "object",
                    "properties": {"order_id": {"type": "string"}},
                    "required": ["order_id"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "search_policy",
                "description": metadata["search_policy"].description,
                "parameters": {
                    "type": "object",
                    "properties": {"query": {"type": "string"}},
                    "required": ["query"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "create_support_ticket",
                "description": metadata["create_support_ticket"].description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "customer_email": {"type": "string"},
                        "subject": {"type": "string"},
                        "body": {"type": "string"},
                    },
                    "required": ["customer_email", "subject", "body"],
                },
            },
        },
    ]
    return ToolRegistry(implementations, schemas, metadata)


REGISTRY = build_tool_registry()
print("Registered tools:", REGISTRY.names())
print("Risk tiers:", {n: REGISTRY.metadata[n].risk.value for n in REGISTRY.names()})

---

## 5. Observability Hooks

Inject tracing at registry and agent boundaries. Every tool call should emit **start**, **end**, and **duration** events to your log backend.

In [ ]:
TRACE_STORE: list[dict[str, Any]] = []


@dataclass
class ObservabilityHooks:
    on_run_start: Callable[[str, str], None] = lambda tid, q: None
    on_run_end: Callable[[str, str, float], None] = lambda tid, ans, ms: None
    on_iteration: Callable[[int], None] = lambda i: None
    on_tool_start: Callable[[str, dict], None] = lambda n, a: None
    on_tool_end: Callable[[str, str, float], None] = lambda n, o, ms: None

    def execute_tool(self, registry: ToolRegistry, name: str, args: dict[str, Any]) -> str:
        fn = registry.get(name)
        if fn is None:
            return _status_error(f"Unknown tool: {name}")
        self.on_tool_start(name, args)
        t0 = time.perf_counter()
        try:
            result = fn(**args)
        except Exception as exc:
            result = _status_error(f"{type(exc).__name__}: {exc}")
        ms = (time.perf_counter() - t0) * 1000
        self.on_tool_end(name, result, ms)
        return result


def make_tracing_hooks(store: list[dict[str, Any]]) -> ObservabilityHooks:
    def log(event: str, **kwargs: Any) -> None:
        store.append({
            "ts": datetime.now(timezone.utc).strftime("%H:%M:%S.%f")[:-3],
            "event": event,
            **kwargs,
        })

    return ObservabilityHooks(
        on_run_start=lambda tid, q: log("run_start", thread_id=tid, query=q[:60]),
        on_run_end=lambda tid, ans, ms: log("run_end", thread_id=tid, ms=round(ms, 1)),
        on_iteration=lambda i: log("iteration", step=i),
        on_tool_start=lambda n, a: log("tool_start", tool=n, args=a),
        on_tool_end=lambda n, o, ms: log("tool_end", tool=n, ms=round(ms, 1), preview=o[:50]),
    )


hooks = make_tracing_hooks(TRACE_STORE)
demo = hooks.execute_tool(REGISTRY, "lookup_order", {"order_id": "ORD-1001"})
print(demo)
print("Trace:", TRACE_STORE[-2:])

---

## 6. Cost Guard and Circuit Breaker

Stop runs that exceed iteration limits, tool-call budgets, or estimated cost. This prevents runaway loops from burning tokens and API quota.

In [ ]:
@dataclass
class CostGuard:
    max_iterations: int
    max_tool_calls: int
    max_cost_usd: float
    cost_per_iteration_usd: float = 0.002
    iteration: int = 0
    tool_calls: int = 0
    estimated_cost_usd: float = 0.0
    tripped: bool = False
    reason: str = ""

    def record_iteration(self) -> bool:
        """Returns False if guard tripped."""
        self.iteration += 1
        self.estimated_cost_usd += self.cost_per_iteration_usd
        return self._check()

    def record_tool_calls(self, n: int = 1) -> bool:
        self.tool_calls += n
        return self._check()

    def _check(self) -> bool:
        if self.iteration >= self.max_iterations:
            self.tripped, self.reason = True, "max_iterations"
        elif self.tool_calls >= self.max_tool_calls:
            self.tripped, self.reason = True, "max_tool_calls"
        elif self.estimated_cost_usd >= self.max_cost_usd:
            self.tripped, self.reason = True, "max_cost_usd"
        return not self.tripped


guard_demo = CostGuard(max_iterations=3, max_tool_calls=5, max_cost_usd=0.01)
while guard_demo.record_iteration():
    pass
print(f"Tripped: {guard_demo.tripped} | reason: {guard_demo.reason} | cost: ${guard_demo.estimated_cost_usd:.4f}")

---

## 7. Agent Loop Factory — `build_support_agent()`

Compile the agent loop **once**. The factory returns a callable that runs the ReAct loop with hooks and cost guard wired in.

`OfflineModel` simulates LLM tool-calling for demos. Swap it for a real OpenAI client when `USE_LIVE_LLM = True`.

In [ ]:
class OfflineModel:
    """Rule-based LLM stand-in — emits tool_calls then a final answer."""

    def __call__(self, messages: list[dict[str, Any]]) -> dict[str, Any]:
        tool_msgs = [m for m in messages if m.get("role") == "tool"]
        user_msg = next(m["content"] for m in messages if m["role"] == "user")

        if not tool_msgs:
            calls = []
            if "ord" in user_msg.lower():
                match = re.search(r"ORD-\d{4}", user_msg.upper())
                oid = match.group(0) if match else "ORD-1001"
                calls.append({
                    "id": "c1", "type": "function",
                    "function": {"name": "lookup_order", "arguments": json.dumps({"order_id": oid})},
                })
            if any(w in user_msg.lower() for w in ("return", "policy", "cancel", "ship")):
                calls.append({
                    "id": "c2", "type": "function",
                    "function": {"name": "search_policy", "arguments": json.dumps({"query": user_msg[:40]})},
                })
            if "ticket" in user_msg.lower() or "escalate" in user_msg.lower():
                calls.append({
                    "id": "c3", "type": "function",
                    "function": {
                        "name": "create_support_ticket",
                        "arguments": json.dumps({
                            "customer_email": "customer@shop.com",
                            "subject": "Escalation from support agent",
                            "body": user_msg[:200],
                        }),
                    },
                })
            if not calls:
                calls.append({
                    "id": "c0", "type": "function",
                    "function": {"name": "search_policy", "arguments": json.dumps({"query": "shipping"})},
                })
            return {"role": "assistant", "content": None, "tool_calls": calls}

        snippets = [m["content"][:80] for m in tool_msgs]
        return {
            "role": "assistant",
            "content": f"Based on ShopCo records: {' | '.join(snippets)}",
        }


@dataclass
class AgentLoop:
    model: Callable[[list[dict[str, Any]]], dict[str, Any]]
    registry: ToolRegistry
    hooks: ObservabilityHooks
    cfg: AppSettings

    def invoke(self, question: str, thread_id: str) -> dict[str, Any]:
        guard = CostGuard(
            max_iterations=self.cfg.max_iterations,
            max_tool_calls=self.cfg.max_tool_calls_per_run,
            max_cost_usd=self.cfg.max_cost_usd,
            cost_per_iteration_usd=self.cfg.cost_per_iteration_usd,
        )
        messages: list[dict[str, Any]] = [
            {"role": "system", "content": "ShopCo support agent. Use tools. Be concise. Cite policy ids."},
            {"role": "user", "content": question},
        ]
        trace: list[str] = []

        while guard.record_iteration():
            self.hooks.on_iteration(guard.iteration)
            response = self.model(messages)
            messages.append(response)
            trace.append(f"model_step={guard.iteration}")

            tool_calls = response.get("tool_calls") or []
            if not tool_calls:
                return {
                    "answer": response.get("content", ""),
                    "messages": messages,
                    "trace": trace,
                    "iterations": guard.iteration,
                    "tool_calls": guard.tool_calls,
                    "cost_usd": round(guard.estimated_cost_usd, 4),
                    "blocked": False,
                }

            if not guard.record_tool_calls(len(tool_calls)):
                break

            for tc in tool_calls:
                name = tc["function"]["name"]
                args = json.loads(tc["function"]["arguments"])
                observation = self.hooks.execute_tool(self.registry, name, args)
                messages.append({"role": "tool", "tool_call_id": tc["id"], "content": observation})
                trace.append(f"tool:{name}")

        return {
            "answer": f"Run stopped: {guard.reason}",
            "messages": messages,
            "trace": trace,
            "iterations": guard.iteration,
            "tool_calls": guard.tool_calls,
            "cost_usd": round(guard.estimated_cost_usd, 4),
            "blocked": True,
            "block_reason": guard.reason,
        }


def build_support_agent(
    cfg: AppSettings,
    registry: ToolRegistry,
    hooks: ObservabilityHooks | None = None,
    model: Callable | None = None,
) -> AgentLoop:
    """Compile once at startup — not per request."""
    return AgentLoop(
        model=model or OfflineModel(),
        registry=registry,
        hooks=hooks or ObservabilityHooks(),
        cfg=cfg,
    )


SUPPORT_AGENT = build_support_agent(settings, REGISTRY, hooks=hooks)
print("Agent compiled:", type(SUPPORT_AGENT).__name__)

---

## 8. Service Layer — `ShopCoAgentService`

HTTP handlers and background workers call a **stable service contract** — not the raw agent loop. This keeps routing, auth, and rate limits outside the graph.

In [ ]:
@dataclass
class AgentResponse:
    answer: str
    thread_id: str
    trace: list[str]
    iterations: int
    tool_calls: int
    cost_usd: float
    blocked: bool = False
    block_reason: str = ""


@dataclass
class RateLimiter:
    max_per_window: int
    window_seconds: float = 60.0
    _timestamps: list[float] = field(default_factory=list)

    def allow(self) -> bool:
        now = time.time()
        self._timestamps = [t for t in self._timestamps if now - t < self.window_seconds]
        if len(self._timestamps) >= self.max_per_window:
            return False
        self._timestamps.append(now)
        return True


class ShopCoAgentService:
    def __init__(
        self,
        agent: AgentLoop,
        cfg: AppSettings,
        hooks: ObservabilityHooks | None = None,
    ):
        self._agent = agent
        self._cfg = cfg
        self._hooks = hooks or ObservabilityHooks()
        self._rate_limiter = RateLimiter(max_per_window=cfg.rate_limit_per_minute)

    def run_support_agent(
        self,
        question: str,
        *,
        thread_id: str | None = None,
        user_tier: str = "standard",
    ) -> AgentResponse:
        thread_id = thread_id or str(uuid.uuid4())

        if not self._rate_limiter.allow():
            return AgentResponse(
                answer="Rate limit exceeded. Please try again shortly.",
                thread_id=thread_id,
                trace=["rate_limited"],
                iterations=0,
                tool_calls=0,
                cost_usd=0.0,
                blocked=True,
                block_reason="rate_limit",
            )

        t0 = time.perf_counter()
        self._hooks.on_run_start(thread_id, question)
        result = self._agent.invoke(question, thread_id)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        self._hooks.on_run_end(thread_id, result["answer"], elapsed_ms)

        return AgentResponse(
            answer=result["answer"],
            thread_id=thread_id,
            trace=result["trace"],
            iterations=result["iterations"],
            tool_calls=result["tool_calls"],
            cost_usd=result["cost_usd"],
            blocked=result.get("blocked", False),
            block_reason=result.get("block_reason", ""),
        )


service = ShopCoAgentService(SUPPORT_AGENT, settings, hooks=hooks)
print("Service ready:", type(service).__name__)

---

## 9. HTTP API Layer — Thin Routes

In production you wrap the service with FastAPI or similar. Here we simulate the HTTP contract as Python request/response objects — no web server required.

In [ ]:
@dataclass
class ChatRequest:
    question: str
    thread_id: str | None = None


@dataclass
class ChatResponse:
    answer: str
    thread_id: str
    trace: list[str]
    iterations: int
    tool_calls: int
    cost_usd: float
    blocked: bool


class SupportChatAPI:
    """Simulates L7 HTTP layer — maps to FastAPI routes in production."""

    def __init__(self, service: ShopCoAgentService):
        self._service = service
        self._started_at = datetime.now(timezone.utc)
        self._warm = False

    def health(self) -> dict[str, Any]:
        return {
            "status": "ok",
            "uptime_seconds": (datetime.now(timezone.utc) - self._started_at).total_seconds(),
            "warmed": self._warm,
        }

    def warmup(self) -> dict[str, Any]:
        t0 = time.perf_counter()
        _ = self._service.run_support_agent("ping", thread_id="warmup")
        ms = (time.perf_counter() - t0) * 1000
        self._warm = True
        return {"status": "warm", "ms": round(ms, 1)}

    def chat(self, req: ChatRequest) -> ChatResponse:
        resp = self._service.run_support_agent(req.question, thread_id=req.thread_id)
        return ChatResponse(
            answer=resp.answer,
            thread_id=resp.thread_id,
            trace=resp.trace,
            iterations=resp.iterations,
            tool_calls=resp.tool_calls,
            cost_usd=resp.cost_usd,
            blocked=resp.blocked,
        )


api = SupportChatAPI(service)
print("Health:", api.health())
print("Warmup:", api.warmup())

---

## 10. Tracing Configuration

Production deployments send traces to LangSmith, Langfuse, or your own log pipeline. Configure via environment — the service tags every run with `thread_id` and `user_tier`.

In [ ]:
TRACING_CONFIG = {
    "enabled": settings.tracing_enabled,
    "project": settings.tracing_project,
    "tags": ["shopco-support", "production"],
    "metadata_keys": ["thread_id", "user_tier", "iterations", "cost_usd"],
    # In production, set these env vars on your deployment platform:
    "env_vars": {
        "LANGCHAIN_TRACING_V2": "true",
        "LANGCHAIN_API_KEY": "<from-secret-manager>",
        "LANGCHAIN_PROJECT": settings.tracing_project,
    },
}

print("Tracing config:")
print(pretty({k: v for k, v in TRACING_CONFIG.items() if k != "env_vars"}))

---

## 11. End-to-End Production Demo

Full stack: HTTP → service → agent loop → tools → trace.

In [ ]:
TRACE_STORE.clear()

queries = [
    "Where is order ORD-1001 and what is your return policy?",
    "Please escalate this issue and create a ticket for billing problems.",
]

for q in queries:
    resp = api.chat(ChatRequest(question=q))
    print(f"\nQ: {q}")
    print(f"A: {resp.answer[:120]}...")
    print(f"   thread={resp.thread_id[:8]}... | iter={resp.iterations} | tools={resp.tool_calls} | cost=${resp.cost_usd}")
    print(f"   trace: {resp.trace}")

print(f"\n--- Trace store ({len(TRACE_STORE)} events) ---")
for e in TRACE_STORE[-8:]:
    print(f"  {e['ts']} {e['event']:<12} { {k: v for k, v in e.items() if k not in ('ts', 'event')} }")

---

## 12. Application Bootstrap — Startup Sequence

Wire everything once when the process starts:

In [ ]:
def create_app() -> SupportChatAPI:
    """Production startup — call once per worker process."""
    cfg = AppSettings.from_env()
    trace_store: list[dict[str, Any]] = []
    obs_hooks = make_tracing_hooks(trace_store)
    registry = build_tool_registry()
    agent = build_support_agent(cfg, registry, hooks=obs_hooks)
    svc = ShopCoAgentService(agent, cfg, hooks=obs_hooks)
    app = SupportChatAPI(svc)
    app.warmup()
    return app


prod_app = create_app()
print("Bootstrap complete:", prod_app.health())

---

## 13. Deployment Checklist

| # | Item |
|---|------|
| 1 | `build_tool_registry()` called once at startup |
| 2 | Secrets loaded from environment / secret manager only |
| 3 | `max_iterations` and cost guard configured |
| 4 | Structured trace logging to log backend |
| 5 | Tracing enabled (LangSmith, Langfuse, or equivalent) |
| 6 | `GET /health` returns ok; warmup runs on startup |
| 7 | Rate limits on agent chat endpoint |
| 8 | HITL gate for write/delete tools (if enabled) |
| 9 | Eval suite passes baseline (≥10 golden prompts) |
| 10 | Agent graph version pinned; rollback documented |

In [ ]:
DEPLOYMENT_CHECKLIST = [
    "Tool registry compiled at startup",
    "Secrets from environment only",
    "max_iterations and cost guard configured",
    "Structured traces to log backend",
    "Tracing project configured",
    "Health endpoint returns ok",
    "Rate limits on chat endpoint",
    "HITL gate for write/delete tools",
    "Eval suite passes baseline",
    "Graph version pinned and rollback documented",
]

for i, item in enumerate(DEPLOYMENT_CHECKLIST, 1):
    print(f"{i:2}. [ ] {item}")

---

## 14. Extension Points

| Extension | Where to add |
|-----------|--------------|
| Write tools + HITL | Wrap `hooks.execute_tool` with approval queue |
| MCP tools | Replace L2 integrations with MCP client calls |
| Supervisor MAS | Swap single agent for supervisor + worker subgraphs |
| Session memory | Persist `messages` by `thread_id` in a store |
| Intent routing | Pre-process user query before agent loop |
| Vector RAG | Replace keyword `search_policy` with retriever tool |

---

## 15. Optional — Live OpenAI Integration

Set `USE_LIVE_LLM = True` and provide a valid `OPENAI_API_KEY` to swap `OfflineModel` for the real API.

In [ ]:
def make_live_model(cfg: AppSettings):
    from openai import OpenAI

    client = OpenAI(api_key=cfg.openai_api_key)
    schemas = REGISTRY.schemas

    def call_model(messages: list[dict[str, Any]]) -> dict[str, Any]:
        api_messages = []
        for m in messages:
            if m["role"] == "tool":
                api_messages.append({
                    "role": "tool",
                    "tool_call_id": m["tool_call_id"],
                    "content": m["content"],
                })
            else:
                api_messages.append({"role": m["role"], "content": m.get("content")})
        resp = client.chat.completions.create(
            model=cfg.default_model,
            messages=api_messages,
            tools=schemas,
            temperature=0,
        )
        msg = resp.choices[0].message
        out: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            out["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in msg.tool_calls
            ]
        return out

    return call_model


if USE_LIVE_LLM:
    live_agent = build_support_agent(settings, REGISTRY, hooks=hooks, model=make_live_model(settings))
    live_service = ShopCoAgentService(live_agent, settings, hooks=hooks)
    live_resp = live_service.run_support_agent("Where is order ORD-1001?")
    print(live_resp.answer)
else:
    print("Offline mode — set USE_LIVE_LLM = True for live OpenAI demo.")

---

## 16. Check Your Understanding

1. Why compile `build_tool_registry()` and `build_support_agent()` at **startup**, not per request?
2. What belongs in the **service layer** vs the **HTTP layer**?
3. Name three conditions that should **trip the cost guard**.
4. What events should observability hooks emit for every tool call?
5. Why run a **warmup** invoke on startup?
6. Where do you add HITL approval for write tools?
7. What should every deployment checklist item verify before go-live?

---

## 17. Summary

Production tool use is more than a working agent loop:

| Pattern | Implementation |
|---------|----------------|
| Registry factory | `build_tool_registry()` — schemas + risk metadata |
| Agent factory | `build_support_agent()` — compile once |
| Service wrapper | `ShopCoAgentService.run_support_agent()` |
| HTTP layer | `SupportChatAPI` — thin `/health` and `/chat` |
| Observability | `ObservabilityHooks` + `TRACE_STORE` |
| Safety | `CostGuard` + `RateLimiter` |
| Bootstrap | `create_app()` — wire once per worker |

The ShopCo demo ran the full stack offline: registry → agent loop → tools → traces → simulated HTTP. That is the blueprint you extend with real APIs, HITL gates, supervisor graphs, and live LLM providers when you deploy.